# 초구 예측 — model_12 (3단계: 위치 1단계 보완 = 모든 카운트 위치 성향)

- 위치 1단계 보완 (README 5장 3번, 4-23·4-24): 시즌 초 배수는 model_11에서 채택
- 0장 0-1: 남은 세 후보 크기 (2019, 교차 이득 ×1000) -> 모든 카운트 +1.47 / 투수 전체 성향 +0.26 (모든 카운트 위에서는 +0.1) / 분산 따로 −0.08
- 모델: model_11에서 **위치 기준점에 모든 카운트 위치 성향만** 더함 (사용자 결정: 모든 카운트부터, 나머지 둘은 아직 [x] 안 함)
  - 같은 투수·구종·손의 0-0 아닌 카운트 공에서, 카운트 효과를 뺀 위치 치우침 δ를 전날까지 누적 -> 기준점 평균을 δ만큼 옮김 (분산 그대로)
  - 위치 파라미터(반감기·k_hand)와 모든 카운트 파라미터를 함께 2019로 다시 고름 (실제 구종 기준 24칸 log loss)
  - 구종(model_09)은 그대로 -> 실전 변화 = 순수한 위치 개선 효과
- 비교 대상: model_11

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):  # model/ 폴더에서 열면 한 단계 위(데이터·fp_common·results 있는 곳)로
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc

MODEL = "model_12"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

초구 50,709개 (검증 13,458 / 평가 12,615)


## 0. 모델 전 확인

### 0-1. 위치 1단계 보완 남은 후보 크기

- 출발: model_11 위치 (기준점 decay 173구 -> 투수×구종 k_type 1000 -> 투수×구종×손 k_hand 17.3, 반감 42.1, 시즌 초 배수 0.707)
- 후보마다 model_11 위에 하나만 얹음, 후보 자기 파라미터 + k_hand를 작은 격자로 고름 (k_type은 548–3000 평평이라 1000 고정, 반감기·시즌 초 배수는 model_11 값)
- 기준: 2019 실제 구종 기준 24칸 log loss (README 3장), 2020은 보지 않음
- 이득 ×1000 = model_11 − 후보 (양수 = 좋아짐), 두 가지
  - 2019 1등: 격자 1등의 이득 (고르는 데 쓴 2019로 잼 -> 조금 부풀려짐)
  - 교차: 2019 경기를 5겹으로 나눠, 나머지 4겹 1등 조합으로 그 겹을 채점 (고르는 몫을 뺀 정직한 크기, 4-15·4-21와 같은 방식)
  - 95% 구간은 경기 단위 부트스트랩
- 새 표는 모두 `fc.prior_table` + `fc.lookup` (전날까지 트래킹, 같은 날 제외) -> 미래 정보 없음

#### 0-1-0. 공통: model_11 위치 재현과 채점 도구

- 평균만 옮기기 `shift_mean`: 기준점 모멘트(x, zn, xx, zz, xz)의 평균만 (dx, dz)만큼 옮기고 분산·공분산은 그대로 (xx = 분산 + 평균²이라 같이 고침)

In [2]:
DIAG = "model_11"
pp15 = fc.load_results([DIAG])[DIAG]  # 재현 확인용
HL15, KT15, KH15, SF15 = 42.1, 1000.0, 17.3, 0.707  # model_11 위치: 반감기, k_type, k_hand, 시즌 초 배수
REF_HL = 173.0  # 기준점 반감기 (분석 대상 투수 구종×손, 공 수)
ZM = 32  # 24칸 적분 점 수

L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 분석 대상 투수 구종×손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29) (model_11와 같음)
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_sn(q, half_life, season_f, cache=True):  # 투수×구종, 투수×구종×손 전날까지 위치 합계 S·개수 N 두 쌍 (model_11와 같음)
    return [fc.lookup(fc.prior_table(fp, keys, L_COLS, half_life, season_f=season_f, cache=cache), q, keys, L_COLS, season_f=season_f)
            for keys in (K1, K2)]

i19 = np.arange(len(q_val))
yz19 = q_val["zone"].map(fc.ZONE_IDX).to_numpy()  # 2019 실제 칸 번호
gid19 = q_val["gameID"].to_numpy()
REF19 = loc_league(q_val, fc.VAL_YEAR, "decay", REF_HL)  # 2019 기준점 (n, 29)
SN19 = loc_sn(q_val, HL15, SF15)  # 2019 투수×구종·투수×구종×손 S·N (model_11 값)

def chain(P, k_type=KT15, k_hand=KH15):  # 출발점 P (n, 29) -> 투수×구종 (k_type) -> 투수×구종×손 (k_hand) (model_11 당김)
    (S1, N1), (S2, N2) = SN19
    return fc.shrink(S2, N2, fc.shrink(S1, N1, P, k_type), k_hand)

def shift_mean(M, dx, dz):  # 모멘트 M (n, 29)의 평균만 (dx, dz) 옮김, 분산·공분산 그대로 (24칸 원-핫 열은 정규분포에 안 쓰여 그대로)
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    nx, nz = mx + dx, mz + dz
    M[:, 2] += nx ** 2 - mx ** 2  # E[x²] = 분산 + 평균²
    M[:, 3] += nz ** 2 - mz ** 2
    M[:, 4] += nx * nz - mx * mz  # E[x·z] = 공분산 + 평균 곱
    M[:, 0], M[:, 1] = nx, nz
    return M

def ll_of(M):  # 모멘트 (n, 29) -> 2019 공별 24칸 log loss (실제 구종)
    P = fc.zone_from_gauss_cov(fc.gauss(M, q_val), q_val, ZM)
    return -np.log(np.clip(P[i19, yz19], 1e-12, None))

LL15 = ll_of(chain(REF19))
assert np.allclose(LL15, fc.align(pp15, fp, fc.VAL_YEAR)["ll_zone_true"].to_numpy()), "model_11 위치 재현 실패"
print(f"model_11 2019 24칸 log loss (실제 구종) {LL15.mean():.5f} (재현 확인)")

# 채점 도구
games = np.unique(gid19)
FOLD = pd.Series(np.random.default_rng(0).permutation(len(games)) % 5, index=games).reindex(gid19).to_numpy()  # 경기 단위 5겹

def game_ci(d, B=1000):  # 경기 단위 부트스트랩 95% 구간
    s = pd.DataFrame({"g": gid19, "d": d}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])

def cv_ll(LL):  # 5겹 교차: 나머지 4겹에서 1등인 조합으로 그 겹 채점 -> 공별 log loss (n,)
    out = np.empty(LL.shape[1])
    for f in range(5):
        b = LL[:, FOLD != f].mean(1).argmin()
        out[FOLD == f] = LL[b, FOLD == f]
    return out

RES = {}  # 후보 이름 -> (조합 표, 공별 log loss (조합, n))

def run_grid(name, grid, fn):  # grid 모든 조합을 fn(**조합) -> 공별 log loss로 채점, 1등 순 표
    combos = pd.MultiIndex.from_product(grid.values(), names=list(grid)).to_frame(index=False)
    recs = combos.astype(object).to_dict("records")
    LL = np.stack([fn(**r) for r in recs])
    tab = combos.assign(logloss=LL.mean(1), **{"이득 ×1000": (LL15.mean() - LL.mean(1)) * 1000})
    RES[name] = (tab, LL)
    top = tab.sort_values("logloss")
    edge = [k for k in grid if len(grid[k]) > 1 and top.iloc[0][k] in (grid[k][0], grid[k][-1])]  # 1등이 후보 범위 끝인 파라미터
    print(f"{name}: 조합 {len(tab)}개, 1등 {top.iloc[0][list(grid)].to_dict()}, 범위 끝: {edge or '없음'}")
    display(top.head(6).round(5))
    return top

def summary_row(name):  # 후보 하나 요약: 2019 1등 이득과 교차 이득 (×1000, 95% 구간)
    tab, LL = RES[name]
    b = tab["logloss"].to_numpy().argmin()
    d1, dc = LL15 - LL[b], LL15 - cv_ll(LL)
    (l1, h1), (lc, hc) = game_ci(d1), game_ci(dc)
    return {"1등 파라미터": str({k: v for k, v in tab.iloc[b].items() if k not in ("logloss", "이득 ×1000")}),
            "2019 log loss": LL[b].mean(), "1등 이득": d1.mean() * 1000, "1등 95%": f"{l1 * 1000:+.2f} ~ {h1 * 1000:+.2f}",
            "교차 이득": dc.mean() * 1000, "교차 95%": f"{lc * 1000:+.2f} ~ {hc * 1000:+.2f}"}

model_11 2019 24칸 log loss (실제 구종) 2.89276 (재현 확인)


#### 0-1-1. 투수 전체 성향 (구종 무관)

- 지금: 기록 적은 구종은 기준점(분석 대상 투수 그 구종×손 평균) 쪽으로 감
- 바꿈: 기준점의 평균을 "그 투수가 구종과 상관없이 기준점보다 치우친 만큼" δ 옮김 -> 그 투수의 다른 구종 초구를 빌려 씀
  - δ = 그 투수 초구들의 (실제 위치 − 그 공의 기준점) 전날까지 반감기 가중 합 / (개수 + k_p) -> 기록 적으면 0 쪽 (구종 구성 차이는 잔차라 걸러짐)
  - 평균(좌우 x, 높이 비율 zn)만 옮김, 분산은 그대로
- 파라미터: δ 묶음(투수 / 투수×손), δ 반감기 hl_p (초구 수), δ 당김 k_p, k_hand / δ 표에도 시즌 초 배수 0.707

In [3]:
# 모든 초구의 기준점 (잔차 계산용): 해마다 그 해 기준 (2017 첫날들은 지난 기록이 없어 빈 값 -> 잔차 빈 값, prior_table이 셈에서 뺌)
REF_ALL = np.full((len(fp), 2), np.nan)
for yr in range(int(fp["GYEAR"].min()), fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    REF_ALL[k] = loc_league(fp[k], yr, "decay", REF_HL)[:, :2]
FP_RES = fp[["pitcher_code", "pitch_type", "same_side", "game_date", "gameID", "pit_no"]].assign(
    rx=fp["x"].to_numpy() - REF_ALL[:, 0], rz=fp["zn"].to_numpy() - REF_ALL[:, 1])  # 초구 위치 − 그 공 기준점 (좌우 cm, 높이 비율)
print("초구 잔차 평균 (전체 0 근처여야 함):", FP_RES[["rx", "rz"]].mean().round(4).to_dict())

PKEYS = {"투수": ["pitcher_code"], "투수×손": ["pitcher_code", "same_side"]}
_DA = {}
def delta_a(keys, hl_p):  # 2019 공마다 투수(×손) 잔차 합 S (n, 2)·개수 N (n, 2) (전날까지, 시즌 초 배수)
    if (keys, hl_p) not in _DA:
        ks = PKEYS[keys]
        _DA[(keys, hl_p)] = fc.lookup(fc.prior_table(FP_RES, ks, ["rx", "rz"], hl_p, season_f=SF15, cache=False), q_val, ks, ["rx", "rz"], season_f=SF15)
    return _DA[(keys, hl_p)]

def ll_a(keys, hl_p, k_p, k_hand):
    S, N = delta_a(keys, hl_p)
    d = S / (N + k_p)  # 투수 치우침 (0 쪽으로 k_p만큼 당김)
    return ll_of(chain(shift_mean(REF19, d[:, 0], d[:, 1]), k_hand=k_hand))

GRID_A = {"keys": ["투수", "투수×손"], "hl_p": [100.0, 300.0, 1000.0], "k_p": [10.0, 30.0, 100.0, 300.0], "k_hand": [13.2, 17.3, 22.8]}
TOP_A = run_grid("투수 전체 성향", GRID_A, ll_a)

초구 잔차 평균 (전체 0 근처여야 함): {'rx': 0.1662, 'rz': 0.0043}
투수 전체 성향: 조합 72개, 1등 {'keys': '투수', 'hl_p': 300.0, 'k_p': 300.0, 'k_hand': 17.3}, 범위 끝: ['keys', 'k_p']


,keys,hl_p,k_p,k_hand,logloss,이득 ×1000
22,투수,300.0,300.0,17.3,2.89228,0.48182
34,투수,1000.0,300.0,17.3,2.89231,0.44775
7,투수,100.0,100.0,17.3,2.89231,0.44401
10,투수,100.0,300.0,17.3,2.89235,0.40282
19,투수,300.0,100.0,17.3,2.89236,0.39830
31,투수,1000.0,100.0,17.3,2.89246,0.29877


#### 0-1-2. 모든 카운트 위치 성향

- 지금 위치는 초구 기록만 씀 -> 같은 투수·구종·손의 **0-0이 아닌 카운트** 공을 빌려 씀 (구종 의심 공 제외)
- 카운트마다 위치가 다름 (2스트라이크면 존 밖이 많음) -> 카운트 차이를 걸러야 함
  - 공마다 잔차 = 위치 − 분석 대상 투수 그 카운트×구종×손 전날까지 평균 (반감 300구)
  - δ = 투수×구종×손 잔차 전날까지 반감기 가중 합 / (개수 + k_b) -> 기준점 평균을 δ만큼 옮김 (평균만, 분산 그대로)
  - = "이 투수는 이 구종을 다른 카운트에서 리그보다 얼마나 치우치게 던지나"를 초구에도 적용
- 파라미터: δ 묶음(투수×구종×손 / 투수×구종), 반감기 hl_b (다른 카운트 공 수), k_b, k_hand / 시즌 초 배수 0.707

In [4]:
MO = m[~m["label_doubt"] & ~(m["ball_before"].eq(0) & m["strike_before"].eq(0))].copy()  # 0-0 아닌 공 (구종 의심 제외)
MO["x"], MO["zn"] = MO["plate_x_in_CM"], MO["z_norm"]
MO["cnt"] = MO["ball_before"].astype(int).astype(str) + "-" + MO["strike_before"].astype(int).astype(str)  # 카운트
CK = ["cnt", "pitch_type", "same_side"]
S, N = fc.lookup(fc.prior_table(MO, CK, ["x", "zn"], 300.0, cache=False), MO, CK, ["x", "zn"])  # 리그 카운트×구종×손 전날까지 평균 (반감 300구)
ref_c = np.where(N > 0, S / np.maximum(N, 1e-12), np.nan)
MO["rx"], MO["rz"] = MO["x"].to_numpy() - ref_c[:, 0], MO["zn"].to_numpy() - ref_c[:, 1]  # 카운트 효과를 뺀 잔차
print(f"0-0 아닌 공 {len(MO):,}개, 잔차 평균", MO[["rx", "rz"]].mean().round(4).to_dict())
print("카운트별 리그 평균 높이 비율 (카운트 효과가 큰지):", MO.groupby("cnt")["zn"].mean().round(3).to_dict())

BKEYS = {"투수×구종×손": K2, "투수×구종": K1}
_DB = {}
def delta_b(keys, hl_b):  # 2019 공마다 다른 카운트 잔차 합 S·개수 N (n, 2)
    if (keys, hl_b) not in _DB:
        ks = BKEYS[keys]
        _DB[(keys, hl_b)] = fc.lookup(fc.prior_table(MO, ks, ["rx", "rz"], hl_b, season_f=SF15, cache=False), q_val, ks, ["rx", "rz"], season_f=SF15)
    return _DB[(keys, hl_b)]

def ll_b(keys, hl_b, k_b, k_hand):
    S, N = delta_b(keys, hl_b)
    d = S / (N + k_b)
    return ll_of(chain(shift_mean(REF19, d[:, 0], d[:, 1]), k_hand=k_hand))

GRID_B = {"keys": ["투수×구종×손", "투수×구종"], "hl_b": [100.0, 300.0, 1000.0], "k_b": [10.0, 30.0, 100.0, 300.0], "k_hand": [13.2, 17.3, 22.8]}
TOP_B = run_grid("모든 카운트", GRID_B, ll_b)

0-0 아닌 공 146,388개, 잔차 평균 {'rx': 0.2337, 'rz': 0.0121}
카운트별 리그 평균 높이 비율 (카운트 효과가 큰지): {'0-1': 0.371, '0-2': 0.349, '1-0': 0.419, '1-1': 0.389, '1-2': 0.316, '2-0': 0.47, '2-1': 0.432, '2-2': 0.354, '3-0': 0.523, '3-1': 0.477, '3-2': 0.422}
모든 카운트: 조합 72개, 1등 {'keys': '투수×구종×손', 'hl_b': 100.0, 'k_b': 30.0, 'k_hand': 22.8}, 범위 끝: ['keys', 'hl_b', 'k_hand']


,keys,hl_b,k_b,k_hand,logloss,이득 ×1000
5,투수×구종×손,100.0,30.0,22.8,2.89104,1.71426
8,투수×구종×손,100.0,100.0,22.8,2.89110,1.65389
7,투수×구종×손,100.0,100.0,17.3,2.89119,1.57002
4,투수×구종×손,100.0,30.0,17.3,2.89121,1.55053
20,투수×구종×손,300.0,100.0,22.8,2.89145,1.31177
19,투수×구종×손,300.0,100.0,17.3,2.89151,1.25040


#### 0-1-3. 투수 전체 성향 + 모든 카운트 함께

- 다른 카운트 δ를 0 대신 투수 전체 성향 δ_A 쪽으로 당김: δ = (S_B + k_b·δ_A) / (N_B + k_b) -> 다른 카운트 기록이 적으면 투수 전체 성향
- 투수 전체 성향은 0-1-1 1등 값 고정, 모든 카운트 묶음은 0-1-2 1등 묶음 고정, 나머지 다시 고름

In [5]:
bA, bB = TOP_A.iloc[0], TOP_B.iloc[0]
SA, NA = delta_a(bA["keys"], float(bA["hl_p"]))
DA = SA / (NA + float(bA["k_p"]))  # 투수 전체 성향 (0-1-1 1등)

def ll_ab(hl_b, k_b, k_hand):
    S, N = delta_b(bB["keys"], hl_b)
    d = (S + k_b * DA) / (N + k_b)  # 다른 카운트 기록 -> 투수 전체 성향 쪽으로 당김
    return ll_of(chain(shift_mean(REF19, d[:, 0], d[:, 1]), k_hand=k_hand))

GRID_AB = {"hl_b": [100.0, 300.0, 1000.0], "k_b": [10.0, 30.0, 100.0, 300.0], "k_hand": [13.2, 17.3, 22.8]}
print(f"고정: 투수 전체 성향 {bA[list(GRID_A)].to_dict()}, 모든 카운트 묶음 {bB['keys']}")
TOP_AB = run_grid("투수 전체 + 모든 카운트", GRID_AB, ll_ab)

고정: 투수 전체 성향 {'keys': '투수', 'hl_p': 300.0, 'k_p': 300.0, 'k_hand': 17.3}, 모든 카운트 묶음 투수×구종×손
투수 전체 + 모든 카운트: 조합 36개, 1등 {'hl_b': 100.0, 'k_b': 100.0, 'k_hand': 22.8}, 범위 끝: ['hl_b', 'k_hand']


,hl_b,k_b,k_hand,logloss,이득 ×1000
8,100.0,100.0,22.8,2.89094,1.82266
5,100.0,30.0,22.8,2.89102,1.73772
7,100.0,100.0,17.3,2.89111,1.64857
4,100.0,30.0,17.3,2.89121,1.54743
20,300.0,100.0,22.8,2.89134,1.41318
19,300.0,100.0,17.3,2.89147,1.29064


#### 0-1-4. 분산 따로 당기기

- 지금: x²·z²·xz 평균도 x·z와 같이 기준점 쪽으로 당김 -> 기준점 분산에 투수끼리 평균 차이가 섞여 퍼짐에 들어감
- 바꿈: 평균(x, zn)은 지금대로, 분산·공분산은 따로
  - 자기 분산: 투수×구종×손 전날까지 (반감기 hl_var) x, zn, x², zn², x·zn -> 분산·공분산
  - 당김 목표: 구종×손별 **투수 안 분산** (지난 시즌까지 투수×구종×손 분산의 (공 수 − 1) 가중 평균)
  - 분산 = (N·자기 + k_var·투수 안) / (N + k_var), unc 켜면 × (1 + 1/(N + k_hand)) (평균을 덜 알수록 넓힘)
- 파라미터: hl_var, k_var, unc, k_hand / 시즌 초 배수 0.707

In [6]:
MCOLS = ["x", "zn", "xx", "zz", "xz"]  # 위치 모멘트

def within_var(year):  # 구종×손별 투수 안 분산·공분산 (지난 시즌까지) -> 2019 공마다 (n, 3): 좌우 분산, 높이 비율 분산, 공분산
    past = fp[fp["GYEAR"] < year]
    g = past.groupby(K2)
    n = g.size()
    c = pd.DataFrame({"vx": g["x"].var(), "vz": g["zn"].var(), "cxz": g.apply(lambda d: d["x"].cov(d["zn"]) if len(d) > 1 else np.nan)})
    w = (n - 1).clip(lower=0)
    num = (c.fillna(0).mul(w, axis=0)).groupby(level=["pitch_type", "same_side"]).sum()
    v = num.div(w.groupby(level=["pitch_type", "same_side"]).sum(), axis=0)  # (공 수 − 1) 가중 평균
    allv = c.fillna(0).mul(w, axis=0).sum() / w.sum()  # 구종×손 기록 없을 때
    return v.reindex(pd.MultiIndex.from_frame(q_val[KR])).fillna(allv).to_numpy()

V0 = within_var(fc.VAL_YEAR)
M15 = chain(REF19)  # model_11 모멘트 (평균은 그대로 씀)
print("투수 안 SD (구종×손 평균, 좌우 cm / 높이 비율):", np.sqrt(V0[:, :2]).mean(0).round(3), " vs 지금 예측 SD:",
      np.sqrt(np.maximum(M15[:, 2:4] - M15[:, :2] ** 2, 0)).mean(0).round(3))
_DV = {}
def own_mom(hl_var):  # 투수×구종×손 전날까지 모멘트 합 S·개수 N (n, 5)
    if hl_var not in _DV:
        _DV[hl_var] = fc.lookup(fc.prior_table(fp, K2, MCOLS, hl_var, season_f=SF15, cache=False), q_val, K2, MCOLS, season_f=SF15)
    return _DV[hl_var]

def ll_c(hl_var, k_var, unc, k_hand):
    M = chain(REF19, k_hand=k_hand)  # 평균은 지금대로
    mx, mz = M[:, 0], M[:, 1]
    S, N = own_mom(hl_var)
    n = N[:, 0]
    e = S / np.maximum(N, 1e-12)  # 자기 가중 평균 모멘트
    own = np.column_stack([e[:, 2] - e[:, 0] ** 2, e[:, 3] - e[:, 1] ** 2, e[:, 4] - e[:, 0] * e[:, 1]])  # 자기 분산·공분산
    own = np.where(n[:, None] > 0, own, 0.0)
    V = (n[:, None] * own + k_var * V0) / (n[:, None] + k_var)  # 투수 안 분산 쪽으로 당김
    if unc:
        _, (_, N2) = SN19
        V = V * (1 + 1 / (N2[:, :1] + k_hand))  # 평균을 덜 알수록 넓힘
    M2 = M.copy()
    M2[:, 2], M2[:, 3], M2[:, 4] = V[:, 0] + mx ** 2, V[:, 1] + mz ** 2, V[:, 2] + mx * mz  # 분산 -> 모멘트
    return ll_of(M2)

GRID_C = {"hl_var": [42.1, 100.0, 300.0], "k_var": [3.0, 10.0, 30.0, 100.0], "unc": [False, True], "k_hand": [13.2, 17.3, 22.8]}
TOP_C = run_grid("분산 따로", GRID_C, ll_c)

투수 안 SD (구종×손 평균, 좌우 cm / 높이 비율): [24.856  0.516]  vs 지금 예측 SD: [23.701  0.51 ]
분산 따로: 조합 72개, 1등 {'hl_var': 100.0, 'k_var': 10.0, 'unc': False, 'k_hand': 17.3}, 범위 끝: ['unc']


,hl_var,k_var,unc,k_hand,logloss,이득 ×1000
31,100.0,10.0,False,17.3,2.89256,0.20064
32,100.0,10.0,False,22.8,2.89264,0.12086
55,300.0,10.0,False,17.3,2.89269,0.06411
56,300.0,10.0,False,22.8,2.89278,-0.02480
30,100.0,10.0,False,13.2,2.89283,-0.07515
34,100.0,10.0,True,17.3,2.89292,-0.16617


#### 0-1-5. 요약

- 이득 ×1000 = model_11 − 후보 (양수 = 좋아짐), 교차 = 고르는 몫을 뺀 크기
- 겹침: "투수 전체 + 모든 카운트" − 큰 쪽 하나 = 작은 쪽을 위에 얹었을 때 남는 몫

In [7]:
SUM = pd.DataFrame({nm: summary_row(nm) for nm in ["투수 전체 성향", "모든 카운트", "투수 전체 + 모든 카운트", "분산 따로"]}).T
display(SUM)
for nm, (tab, _) in RES.items():  # 파라미터별 민감도: 값마다 1등 log loss
    print(nm, {k: tab.groupby(k)["logloss"].min().round(5).to_dict() for k in tab.columns if k not in ("logloss", "이득 ×1000")})

,1등 파라미터,2019 log loss,1등 이득,1등 95%,교차 이득,교차 95%
투수 전체 성향,"{'keys': '투수', 'hl_p': np.float64(300.0), 'k_p...",2.892276,0.481817,-0.03 ~ +1.03,0.26043,-0.36 ~ +0.89
모든 카운트,"{'keys': '투수×구종×손', 'hl_b': np.float64(100.0),...",2.891043,1.71426,+0.37 ~ +3.13,1.466544,+0.20 ~ +2.79
투수 전체 + 모든 카운트,"{'hl_b': 100.0, 'k_b': 100.0, 'k_hand': 22.8}",2.890935,1.822657,+0.94 ~ +2.82,1.56539,+0.42 ~ +2.82
분산 따로,"{'hl_var': np.float64(100.0), 'k_var': np.floa...",2.892557,0.200644,-1.37 ~ +1.66,-0.077838,-1.63 ~ +1.44


투수 전체 성향 {'keys': {'투수': 2.89228, '투수×손': 2.89254}, 'hl_p': {100.0: 2.89231, 300.0: 2.89228, 1000.0: 2.89231}, 'k_p': {10.0: 2.89288, 30.0: 2.89262, 100.0: 2.89231, 300.0: 2.89228}, 'k_hand': {13.2: 2.89259, 17.3: 2.89228, 22.8: 2.89251}}
모든 카운트 {'keys': {'투수×구종': 2.89191, '투수×구종×손': 2.89104}, 'hl_b': {100.0: 2.89104, 300.0: 2.89145, 1000.0: 2.89173}, 'k_b': {10.0: 2.89199, 30.0: 2.89104, 100.0: 2.8911, 300.0: 2.89178}, 'k_hand': {13.2: 2.89177, 17.3: 2.89119, 22.8: 2.89104}}
투수 전체 + 모든 카운트 {'hl_b': {100.0: 2.89094, 300.0: 2.89134, 1000.0: 2.89164}, 'k_b': {10.0: 2.89199, 30.0: 2.89102, 100.0: 2.89094, 300.0: 2.89152}, 'k_hand': {13.2: 2.89176, 17.3: 2.89111, 22.8: 2.89094}}
분산 따로 {'hl_var': {42.1: 2.89304, 100.0: 2.89256, 300.0: 2.89269}, 'k_var': {3.0: 2.89569, 10.0: 2.89256, 30.0: 2.89342, 100.0: 2.89828}, 'unc': {False: 2.89256, True: 2.89292}, 'k_hand': {13.2: 2.89283, 17.3: 2.89256, 22.8: 2.89264}}


#### 🔍 0-1 결과: 모든 카운트가 가장 큼, 투수 전체 성향은 대부분 겹침, 분산 따로는 효과 없음
| 후보 | 2019 1등 이득 ×1000 (95%) | 교차 이득 ×1000 (95%) |
| --- | --- | --- |
| **모든 카운트** | **+1.71** (+0.37 ~ +3.13) | **+1.47** (+0.20 ~ +2.79) |
| 투수 전체 성향 | +0.48 (−0.03 ~ +1.03) | +0.26 (−0.36 ~ +0.89) |
| 투수 전체 + 모든 카운트 | +1.82 (+0.94 ~ +2.82) | +1.57 (+0.42 ~ +2.82) |
| 분산 따로 | +0.20 (−1.37 ~ +1.66) | −0.08 (−1.63 ~ +1.44) |

- 모든 카운트: 투수×구종×손 묶음이 투수×구종보다 좋음 (2.89104 vs 2.89191), 짧은 반감기(100구)·k_b 30–100
  - 1등이 범위 끝 (반감 100 아래쪽, k_hand 22.8 위쪽) -> 모델에서 넓혀 다시 고름
  - 크기: 시즌 초 배수(model_11, −0.0005)의 약 3배
- 투수 전체 성향: 단독도 0과 구별 안 됨, 1등 k_p 300(범위 끝 = 약하게 당길수록 좋음) -> 효과 작음
  - 모든 카운트 위에 더하면 +0.11 (교차 +0.10) -> 남는 몫 거의 없음
- 분산 따로: 교차 −0.08, k_var 10만 겨우 0 근처 (3·30·100은 나빠짐) -> 0-1(4-23) 진단대로 퍼짐은 이미 맞음
  - 투수 안 SD (좌우 24.9cm, 높이 비율 0.52)가 지금 예측 SD (23.7cm, 0.51)와 비슷 -> 기준점 분산에 투수끼리 차이가 크게 섞이지 않았음
- -> 적용 순서: **모든 카운트 (model_12)** -> 투수 전체 성향은 model_12 위 남는 몫이 작아 다시 확인만 / 분산 따로는 적용 안 함 후보


## 모델

In [8]:
# 0장이 모델 변수 이름을 덮어쓰지 않았는지 대비: 설정 상수를 다시 정함
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]


## 1. 파라미터

- 구종은 model_09 값 고정, 위치 출발 값은 model_11 (시즌 초 배수 0.707은 고정) -> 3-1에서 다시 고른 값으로 바꿈

In [9]:
# ① 구종 출발점 B4: model_06 값 (고정) - 반감기·alpha·hand_k, 시즌 초 배수 (기준점 = 지난 시즌 전부)
B4_P = dict(half_life=35.4, alpha=7.5, hand_k=5.48, season_f=0.25)
# ① LightGBM 변수: A 이번 타석 상황 + 앞 타석 초구 결과 원-핫 (fc.FIRST_CATS)
FEAT = fc.A_COLS + ["prev_first_" + c for c in fc.FIRST_CATS]
# ② 위치: 출발 값 = model_11 -> 3-1에서 다시 고름
LOC_REF, LOC_REF_HL = "decay", 173.0  # 기준점: 분석 대상 투수 구종×손, 전날까지 반감 173구 (고정)
LOC_HL, K_TYPE, K_HAND = 42.1, 1000.0, 17.3  # 투수 기록 반감기, 투수×구종 당김 k, 투수×구종×손 당김 k (model_11)
LOC_SEASON_F = 0.707  # 시즌 초 배수 (model_11, 고정): 새 시즌 첫 경기 전에 지난 시즌까지 투수 위치 누적 무게에 곱함
LOC_AC = None  # 모든 카운트 위치 성향 (잔차 기준 반감기, δ 반감기, δ 당김 k), None = 없음 (model_11)
ZONE_M = 32
# ① LightGBM: model_07 값 (고정)
LGB_P = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}
LGB_ROUNDS = 730
RETRAIN = "month"  # 재학습 주기: 그 달 첫 경기 전날까지 데이터로 다시 학습 (3월은 4월에, 10월은 9월에 합침)
# 구종 보정 학습 해
TRAIN_FIRST = 2017

## 2. ① 구종: B4 + LightGBM 월 단위 재학습 (model_09과 같음)

### 2-1. B4 (model_06·07·09과 같음)

In [10]:
def b4_online(q, year, half_life, alpha, hand_k, season_f):  # model_06 B4 (n, 9): 지난 시즌 리그 비율 -> 투수 (alpha) -> 투수×손 (hand_k), 시즌 초 배수
    first = year == fp["GYEAR"].min()  # 첫 시즌(2017): 지난 기록 없음 -> 그 시즌 비율 (학습 데이터 출발점용)
    ref = fp.loc[fp["GYEAR"] == year if first else fp["GYEAR"] < year, CLASSES].mean().to_numpy()
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    S, N, S2, N2 = out
    return fc.shrink(S2, N2, fc.shrink(S, N, ref, alpha), hand_k)

K = len(CLASSES)
P_B4 = np.full((len(fp), K), np.nan)  # 2017–2020
for yr in range(TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P_B4[k] = b4_online(fp[k], yr, **B4_P)
b4_19 = fc.prob_metrics(P_B4[val], y_type[val])["logloss"]
assert abs(b4_19 - 1.09367) < 1e-5, b4_19  # model_06·07 B4의 2019 log loss
print(f"2019 B4 log loss {b4_19:.5f} = model_06·07 B4 (재현 확인)")
m07 = pd.read_parquet("results/model_07/per_pitch.parquet")  # 첫 달 재현 확인용
m09 = pd.read_parquet("results/model_09/per_pitch.parquet")  # 구종 재현 확인용 (구종은 model_09 그대로)
m11 = pd.read_parquet("results/model_11/per_pitch.parquet")  # 비교 대상

2019 B4 log loss 1.09367 = model_06·07 B4 (재현 확인)


### 2-2. 변수 (model_07과 같음)

- A 이번 타석 상황 + 앞 타석 초구 결과, 미래 정보 검사 다시 통과 확인

In [11]:
X_all = pd.concat([fc.situation_features(m, fp), fc.game_flow_features(m, fp)], axis=1)[FEAT]
assert X_all.notna().all().all()
for fn in (fc.situation_features, fc.game_flow_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(TRAIN_FIRST)], n=150)
    assert bad.empty, bad
display(X_all.describe().T[["mean", "min", "max"]].round(3))

미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개


,mean,min,max
pc,33.394,0.0,124.0
tto,1.592,1.0,5.0
starter,0.690,0.0,1.0
inning,4.548,1.0,12.0
out_cn,0.970,0.0,2.0
base1,0.359,0.0,1.0
base2,0.194,0.0,1.0
base3,0.100,0.0,1.0
batorder,4.811,1.0,9.0
sdiff,0.369,-21.0,19.0


### 2-2. 월 단위 재학습과 구종 확률 (model_09과 같음)

- 확인: 구종 확률 = model_09 결과 파일 (재현)

In [12]:
y_all = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
INIT = np.log(np.clip(P_B4, 1e-6, None))
INIT -= INIT.mean(1, keepdims=True)  # 출발점 (공마다 상수는 softmax에 영향 없음)
BASE = dict(objective="multiclass", num_class=K, learning_rate=0.03, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
            verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)  # model_05–07과 같음
yr_all = fp["GYEAR"].to_numpy()
DATE = pd.to_datetime(fp["gameID"].str[:8]).to_numpy()
MON = np.clip(fp["gameID"].str[4:6].astype(int).to_numpy(), 4, 9)  # 재학습 묶음 달: 3월 -> 4월, 10월 -> 9월

def fit_rows(tr):  # 학습 행 tr로 LightGBM (트리 수 고정)
    return lgb.train({**BASE, **LGB_P}, lgb.Dataset(X_all.iloc[tr], y_all[tr], init_score=INIT[tr]), LGB_ROUNDS)

def type_prob(bst, idx):  # B4 출발점 + 트리 보정 -> 구종 확률 (n, 9)
    return np.exp(log_softmax(bst.predict(X_all.iloc[idx], raw_score=True) + INIT[idx], axis=1))

P_TYPE_ALL = np.full((len(fp), K), np.nan)
BST, plan = {}, []
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    iy = np.flatnonzero(yr_all == yr)
    for mon in np.unique(MON[iy]):
        te = iy[MON[iy] == mon]
        start = DATE[te].min()  # 그 달 첫 경기일
        tr = np.flatnonzero((yr_all >= TRAIN_FIRST) & (DATE < start))
        assert DATE[tr].max() < start  # 미래 정보 없음
        BST[(yr, mon)] = bst = fit_rows(tr)
        P_TYPE_ALL[te] = type_prob(bst, te)
        plan.append({"연도": yr, "월": mon, "예측 초구": len(te), "학습 초구": len(tr), "그 시즌 학습 초구": int((yr_all[tr] == yr).sum()),
                     "학습 끝": str(pd.Timestamp(DATE[tr].max()).date()), "예측 시작": str(pd.Timestamp(start).date())})
display(pd.DataFrame(plan).set_index(["연도", "월"]))
P11 = {yr: fc.align(m07, fp, yr)[fc.P_TYPE].to_numpy() for yr in (fc.VAL_YEAR, fc.TEST_YEAR)}  # model_07 구종 확률 (fp 그 해 순서)
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:  # 첫 달 = 시즌 단위 (model_07 재현)
    k = MON[msk] == MON[msk].min()
    assert np.allclose(P_TYPE_ALL[msk][k], P11[yr][k], atol=1e-6)
print("첫 달 구종 확률 = model_07 (재현 확인)")
rows = {}
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    for nm, P in [("B4", P_B4[msk]), ("model_07", P11[yr]), (MODEL, P_TYPE_ALL[msk])]:
        r = fc.prob_metrics(P, y_type[msk])
        rows[(yr, nm)] = {"log loss": r["logloss"], "top1": r["top1"], "top2": r["top2"]}
display(pd.DataFrame(rows).T.round(4))
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    assert np.allclose(P_TYPE_ALL[msk], fc.align(m09, fp, yr)[fc.P_TYPE].to_numpy(), atol=1e-6), f"{yr} 구종 확률이 model_09과 다름"
print("구종 확률 = model_09 (재현 확인)")

예측 초구  학습 초구  그 시즌 학습 초구        학습 끝       예측 시작
연도   월                                                  
2019 4   2568  24636           0  2018-10-14  2019-03-23
     5   2403  27204        2568  2019-04-30  2019-05-01
     6   2493  29607        4971  2019-05-31  2019-06-01
     7   1791  32100        7464  2019-06-30  2019-07-02
     8   2362  33891        9255  2019-07-31  2019-08-01
     9   1841  36253       11617  2019-08-31  2019-09-01
2020 5   2264  38094           0  2019-10-01  2020-05-05
     6   2379  40358        2264  2020-05-31  2020-06-02
     7   2233  42737        4643  2020-06-30  2020-07-01
     8   1985  44970        6876  2020-07-31  2020-08-01
     9   3754  46955        8861  2020-08-30  2020-09-01

첫 달 구종 확률 = model_07 (재현 확인)


log loss    top1    top2
2019 B4          1.0937  0.5279  0.8159
     model_07    1.0621  0.5380  0.8174
     model_12    1.0613  0.5385  0.8177
2020 B4          1.0916  0.5264  0.8179
     model_07    1.0570  0.5340  0.8209
     model_12    1.0561  0.5332  0.8218

구종 확률 = model_09 (재현 확인)


## 3. ② 위치: 모든 카운트 위치 성향

- 0-0 아닌 공 (구종 의심 제외)의 위치에서 **분석 대상 투수 그 카운트×구종×손 전날까지 평균**(반감 `ref_hl`구)을 뺀 잔차 -> 카운트 효과를 걸러 냄
- δ = 투수×구종×손 잔차의 전날까지 반감기(`hl_b`, 다른 카운트 공 수) 가중 합 / (개수 + `k_b`) -> 기록 적으면 0 쪽
- 기준점 평균(좌우 x, 높이 비율 zn)을 δ만큼 옮긴 뒤 지금처럼 투수×구종 -> 투수×구종×손 초구 기록 쪽으로 당김 (분산·공분산은 기준점 그대로)
  - = "이 투수는 이 구종을 다른 카운트에서 리그보다 얼마나 치우치게 던지나"를 초구 출발점에 반영
- δ 표에도 시즌 초 배수 0.707 (지난 시즌 기록 무게)
- 확인: `LOC_AC = None`이면 model_11와 같음 (재현)

In [14]:
# 24칸 적분 함수: fc.zone_from_gauss_cov(g, q, m=32) (model_03부터)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 분석 대상 투수 구종×손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_sn(q, half_life, season_f, cache=True):  # 투수×구종, 투수×구종×손 전날까지 초구 위치 합계 S·개수 N 두 쌍 (시즌 초 배수 반영)
    return [fc.lookup(fc.prior_table(fp, keys, L_COLS, half_life, season_f=season_f, cache=cache), q, keys, L_COLS, season_f=season_f)
            for keys in (K1, K2)]

def shift_mean(M, dx, dz):  # 모멘트 M (n, 29)의 평균만 (dx, dz) 옮김, 분산·공분산 그대로 (24칸 원-핫 열은 정규분포에 안 쓰여 그대로)
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    nx, nz = mx + dx, mz + dz
    M[:, 2] += nx ** 2 - mx ** 2  # E[x²] = 분산 + 평균²
    M[:, 3] += nz ** 2 - mz ** 2
    M[:, 4] += nx * nz - mx * mz  # E[x·z] = 공분산 + 평균 곱
    M[:, 0], M[:, 1] = nx, nz
    return M

# 0-0 아닌 공 (구종 의심 제외): 모든 카운트 위치 성향 재료
MO = m[~m["label_doubt"] & ~(m["ball_before"].eq(0) & m["strike_before"].eq(0))].copy()
MO["x"], MO["zn"] = MO["plate_x_in_CM"], MO["z_norm"]  # 좌우 cm (타자 기준), 높이 비율 (존 아랫선 0, 윗선 1)
MO["cnt"] = MO["ball_before"].astype(int).astype(str) + "-" + MO["strike_before"].astype(int).astype(str)  # 카운트
CK = ["cnt", "pitch_type", "same_side"]  # 카운트 효과 그룹
_MO_RES = {}

def mo_resid(ref_hl):  # 0-0 아닌 공마다 위치 − 분석 대상 투수 그 카운트×구종×손 전날까지 평균 (반감 ref_hl구) -> 잔차 표 (rx 좌우 cm, rz 높이 비율)
    if ref_hl not in _MO_RES:
        S, N = fc.lookup(fc.prior_table(MO, CK, ["x", "zn"], ref_hl, cache=False), MO, CK, ["x", "zn"])
        ref = np.where(N > 0, S / np.maximum(N, 1e-12), np.nan)  # 그 카운트×구종×손 기록이 없으면 빈 값 (셈에서 빠짐)
        _MO_RES[ref_hl] = MO[K2 + ["game_date", "gameID", "pit_no"]].assign(rx=MO["x"].to_numpy() - ref[:, 0], rz=MO["zn"].to_numpy() - ref[:, 1])
    return _MO_RES[ref_hl]

def ac_sn(q, ref_hl, hl_b, season_f, cache=True):  # 공마다 투수×구종×손 다른 카운트 잔차 전날까지 합 S·개수 N (n, 2)
    d = mo_resid(ref_hl)
    return fc.lookup(fc.prior_table(d, K2, ["rx", "rz"], hl_b, season_f=season_f, cache=cache), q, K2, ["rx", "rz"], season_f=season_f)

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None):  # 위치 모멘트 (n, 29): 기준점 (+ 모든 카운트 δ) -> 투수×구종 (k_type) -> 투수×구종×손 (k_hand)
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    if ac is not None:  # ac = (잔차 기준 반감기, δ 반감기, δ 당김 k)
        S, N = ac_sn(q, ac[0], ac[1], season_f)
        d = S / (N + ac[2])  # 다른 카운트 치우침 (0 쪽으로 k_b만큼 당김)
        P = shift_mean(P, d[:, 0], d[:, 1])  # 기준점 평균을 옮김
    (S1, N1), (S2, N2) = loc_sn(q, half_life, season_f)
    P = fc.shrink(S1, N1, P, k_type)  # 투수×구종 쪽으로 k_type만큼
    return fc.shrink(S2, N2, P, k_hand)  # 투수×구종×손 쪽으로 k_hand만큼

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, cov=True):  # 구종이 t일 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)  # 구종만 t로 바꾼 사본 (t: 이름 하나 또는 공마다 배열)
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl, season_f, ac), qt)  # 모멘트 -> 정규분포 (높이는 그 타자 존으로 cm 환산)
    return zone_probs(g, qt, cov), g  # 정규분포를 24칸 위에서 적분 (공분산 반영)

# 재현: ac = None -> model_11 실제 구종 기준 24칸 log loss
yz_v = q_val["zone"].map(fc.ZONE_IDX).to_numpy()
P0, _ = loc_given_type(q_val, fc.VAL_YEAR, q_val["pitch_type"].to_numpy(), LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL, LOC_SEASON_F, None)
ll0 = -np.log(np.clip(P0[np.arange(len(q_val)), yz_v], 1e-12, None))
assert np.allclose(ll0, fc.align(m11, fp, fc.VAL_YEAR)["ll_zone_true"].to_numpy()), "ac = None인데 model_11와 다름"
print(f"ac = None -> 2019 실제 구종 기준 24칸 log loss {ll0.mean():.5f} = model_11 (재현 확인)")
print(f"0-0 아닌 공 {len(MO):,}개 / 카운트별 분석 대상 투수 평균 높이 비율:", MO.groupby("cnt")["zn"].mean().round(3).to_dict())

ac = None -> 2019 실제 구종 기준 24칸 log loss 2.89276 = model_11 (재현 확인)
0-0 아닌 공 146,388개 / 카운트별 분석 대상 투수 평균 높이 비율: {'0-1': 0.371, '0-2': 0.349, '1-0': 0.419, '1-1': 0.389, '1-2': 0.316, '2-0': 0.47, '2-1': 0.432, '2-2': 0.354, '3-0': 0.523, '3-1': 0.477, '3-2': 0.422}


### 3-1. 모든 카운트 + 위치 파라미터 다시 고르기 (2019)

- 연결된 파라미터를 함께: 위치 반감기 × k_hand × δ 반감기 `hl_b` × δ 당김 `k_b` (k_type은 548–3000 평평해 1000, 시즌 초 배수 0.707, δ 묶음은 0장대로 투수×구종×손)
  - 0장 1등이 범위 끝(hl_b 100 아래쪽, k_hand 22.8 위쪽) -> 처음부터 넓게
- 거친 격자 전부 -> 범위 끝이면 넓혀 다시 -> 1등 주변 세밀한 격자 전부 -> 잔차 기준 반감기 `ref_hl` (100 / 300 / 1000) 확인
- 대조군 = model_11 (모든 카운트 없음), 1등 vs 대조군 경기 단위 부트스트랩

In [15]:
def grid_search(score, grid, by="logloss"):  # grid의 모든 조합을 score로 채점 (지표 낮은 순)
    names_ = list(grid)
    combos = pd.MultiIndex.from_product(grid.values(), names=names_).to_frame(index=False)  # 후보의 모든 조합 (한 행 = 한 조합)
    rows = [{**r, **score(**r)} for r in combos.astype(object).where(combos.notna(), None).to_dict("records")]  # NaN -> None으로 되돌려 넘김
    return pd.DataFrame(rows).sort_values(by).reset_index(drop=True)

def finer(cands, best, n=3):
    # 거친 후보 목록에서 best와 양옆 후보 사이를 각각 로그 간격으로 n개씩 나눈 세밀한 후보 (best 가운데, 2n−1개)
    # best가 범위 끝이면 바깥으로 2배 넓힘, best가 None이면 None 그대로
    if best is None:
        return [None]
    nums = sorted(c for c in cands if c is not None)  # None 뺀 숫자 후보
    i = nums.index(best)
    lo = nums[i - 1] if i > 0 else best / 2  # 왼쪽 이웃 (없으면 절반)
    hi = nums[i + 1] if i < len(nums) - 1 else best * 2  # 오른쪽 이웃 (없으면 2배)
    vals = np.r_[np.geomspace(lo, best, n), np.geomspace(best, hi, n)[1:]]  # lo~best, best~hi를 같은 비율 간격으로
    return sorted({float(f"{v:.3g}") for v in vals})  # 유효숫자 3자리로 반올림, 중복 제거

def best_of(table, grid):  # 표 1등 행의 파라미터 dict (NaN -> None, 숫자는 float)
    r = table.iloc[0]
    return {k: (None if pd.isna(r[k]) else float(r[k])) for k in grid}

def marginal(table, grid, by="logloss"):  # 파라미터마다 값별 가장 좋은 by 지표 (나머지는 그 값에서 가장 좋은 조합) -> 어느 파라미터가 민감한지
    out = {}
    for k in grid:
        s = table.assign(**{k: table[k].astype(object).where(table[k].notna(), "None")}).groupby(k, sort=False)[by].min()
        out[k] = s.reindex([("None" if v is None else v) for v in grid[k] if ("None" if v is None else v) in s.index])
    return out

def game_ci(d, gid, B=1000):  # 경기 단위 부트스트랩 95% 구간 (같은 경기 공은 독립이 아님)
    s = pd.DataFrame({"g": gid, "d": d}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])


REF19 = loc_league(q_val, fc.VAL_YEAR, LOC_REF, LOC_REF_HL)  # 2019 기준점 (고정)
gid_v = q_val["gameID"].to_numpy()
_LSN, _ACSN = {}, {}  # 2019 lookup 캐시 (prior_table은 캐시 안 함: 조합이 많아 메모리)

def ll_ac(half_life, k_hand, hl_b, k_b, ref_hl=300.0):  # 2019 공별 24칸 log loss (실제 구종), hl_b None = 모든 카운트 없음
    if half_life not in _LSN:
        _LSN[half_life] = loc_sn(q_val, half_life, LOC_SEASON_F, cache=False)
    (S1, N1), (S2, N2) = _LSN[half_life]
    P = REF19
    if hl_b is not None:
        if (ref_hl, hl_b) not in _ACSN:
            _ACSN[(ref_hl, hl_b)] = ac_sn(q_val, ref_hl, hl_b, LOC_SEASON_F, cache=False)
        S, N = _ACSN[(ref_hl, hl_b)]
        d = S / (N + k_b)
        P = shift_mean(REF19, d[:, 0], d[:, 1])
    g = fc.gauss(fc.shrink(S2, N2, fc.shrink(S1, N1, P, K_TYPE), k_hand), q_val)
    Pz = zone_probs(g, q_val)
    return -np.log(np.clip(Pz[np.arange(len(q_val)), yz_v], 1e-12, None))

def score_ac(half_life, k_hand, hl_b, k_b, ref_hl=300.0):
    return {"logloss": ll_ac(half_life, k_hand, hl_b, k_b, ref_hl).mean()}

LL15 = ll_ac(LOC_HL, K_HAND, None, None)  # 대조군 = model_11
assert np.allclose(LL15, ll0)
GRID_AC = {"half_life": [29.7, 42.1, 59.5], "k_hand": [13.2, 17.3, 22.8, 30.0], "hl_b": [25.0, 50.0, 100.0, 200.0], "k_b": [10.0, 30.0, 100.0, 300.0]}
res = grid_search(score_ac, GRID_AC)  # 거친 격자
tried = {k: list(v) for k, v in GRID_AC.items()}
for rnd in range(3):  # 1등이 범위 끝이면 바깥으로 넓힘
    b = best_of(res, GRID_AC)
    add = {k: [v / 2] if v == min(tried[k]) else [v * 2] for k, v in b.items() if v in (min(tried[k]), max(tried[k]))}
    if not add:
        break
    print(f"{rnd + 1}번째 넓히기:", add)
    for k, v in add.items():
        tried[k] += v
    res = pd.concat([res, grid_search(score_ac, {k: (add[k] if k in add else [b[k]]) for k in GRID_AC})])
    res = res.drop_duplicates(list(GRID_AC)).sort_values("logloss").reset_index(drop=True)
b = best_of(res, GRID_AC)
grid_f = {k: finer(tried[k], b[k]) for k in GRID_AC}  # 1등 주변 세밀한 후보
print("거친 1등", b, "-> 세밀한 후보", grid_f)
res = pd.concat([res, grid_search(score_ac, grid_f)]).drop_duplicates(list(GRID_AC)).sort_values("logloss").reset_index(drop=True)
for rnd in range(4):  # 세밀한 격자 뒤에도 1등이 범위 끝이면 그 파라미터만 바깥으로 넓힘 (나머지는 1등 값)
    b = best_of(res, GRID_AC)
    add = {k: [v / 2] if v == min(tried[k]) else [v * 2] for k, v in b.items() if v in (min(tried[k]), max(tried[k]))}
    if not add:
        break
    print(f"세밀한 격자 뒤 {rnd + 1}번째 넓히기:", add)
    for k, v in add.items():
        tried[k] += v
        res = pd.concat([res, grid_search(score_ac, {**{kk: [vv] for kk, vv in b.items()}, k: v})])
    res = res.drop_duplicates(list(GRID_AC)).sort_values("logloss").reset_index(drop=True)
b = best_of(res, GRID_AC)
REF_HLS = [100.0, 300.0, 1000.0, 3000.0, None]  # 잔차 기준 반감기 (None = 반감 없이 누적)
res_ref = grid_search(score_ac, {**{k: [v] for k, v in b.items()}, "ref_hl": REF_HLS})  # 잔차 기준 반감기 확인
rb = res_ref.iloc[0]["ref_hl"]
rb = None if pd.isna(rb) else float(rb)
if rb != 300.0:  # 기준 반감기가 바뀌면 δ 파라미터(hl_b·k_b)를 그 값에서 1등 양옆으로 다시 확인
    nb_ = lambda k: sorted(set(tried[k]) | set(grid_f[k]))
    near = {k: [v for v in nb_(k) if v == b[k] or v in (max([u for u in nb_(k) if u < b[k]], default=b[k]), min([u for u in nb_(k) if u > b[k]], default=b[k]))] for k in ["hl_b", "k_b"]}
    res_ref = pd.concat([res_ref, grid_search(score_ac, {"half_life": [b["half_life"]], "k_hand": [b["k_hand"]], **near, "ref_hl": [rb]})])
    res_ref = res_ref.drop_duplicates(list(GRID_AC) + ["ref_hl"]).sort_values("logloss").reset_index(drop=True)
print(f"2019 24칸 log loss (실제 구종): 대조군 model_11 {LL15.mean():.5f} / 1등 {res.logloss.iloc[0]:.5f}")
display(res.head(10).round(5))
display(res_ref.round(5))
for k, s in marginal(res, {k: sorted(set(tried[k]) | set(grid_f[k])) for k in GRID_AC}).items():
    print(k, s.round(5).to_dict())  # 파라미터별 값마다 1등 (민감도)
best = res_ref.iloc[0]
d = LL15 - ll_ac(**{k: (None if pd.isna(best[k]) else float(best[k])) for k in list(GRID_AC) + ["ref_hl"]})
lo, hi = game_ci(d, gid_v)
print(f"1등 vs model_11 이득 ×1000: {d.mean() * 1000:.3f} (95% {lo * 1000:.3f} ~ {hi * 1000:.3f})")

1번째 넓히기: {'hl_b': [12.5]}
거친 1등 {'half_life': 42.1, 'k_hand': 22.8, 'hl_b': 25.0, 'k_b': 30.0} -> 세밀한 후보 {'half_life': [29.7, 35.4, 42.1, 50.0, 59.5], 'k_hand': [17.3, 19.9, 22.8, 26.2, 30.0], 'hl_b': [12.5, 17.7, 25.0, 35.4, 50.0], 'k_b': [10.0, 17.3, 30.0, 54.8, 100.0]}
세밀한 격자 뒤 1번째 넓히기: {'hl_b': [6.25]}
2019 24칸 log loss (실제 구종): 대조군 model_11 2.89276 / 1등 2.89030


,half_life,k_hand,hl_b,k_b,logloss
0,42.1,19.9,12.5,17.3,2.89030
1,42.1,22.8,12.5,17.3,2.89034
2,42.1,19.9,17.7,17.3,2.89038
3,50.0,22.8,12.5,17.3,2.89038
4,42.1,19.9,17.7,30.0,2.89038
5,35.4,19.9,12.5,17.3,2.89040
6,42.1,22.8,17.7,17.3,2.89040
7,42.1,22.8,17.7,30.0,2.89040
8,42.1,17.3,12.5,17.3,2.89040
9,50.0,19.9,12.5,17.3,2.89040


,half_life,k_hand,hl_b,k_b,ref_hl,logloss
0,42.1,19.9,12.50,17.3,NaN,2.89006
1,42.1,19.9,12.50,17.3,3000.0,2.89008
2,42.1,19.9,17.70,17.3,None,2.89013
3,42.1,19.9,12.50,17.3,1000.0,2.89014
4,42.1,19.9,17.70,30.0,None,2.89015
5,42.1,19.9,12.50,30.0,None,2.89024
6,42.1,19.9,6.25,17.3,None,2.89025
7,42.1,19.9,12.50,17.3,300.0,2.89030
8,42.1,19.9,6.25,10.0,None,2.89031
9,42.1,19.9,12.50,10.0,None,2.89038


half_life {29.7: 2.89071, 35.4: 2.8904, 42.1: 2.8903, 50.0: 2.89038, 59.5: 2.8906}
k_hand {13.2: 2.89121, 17.3: 2.8904, 19.9: 2.8903, 22.8: 2.89034, 26.2: 2.8905, 30.0: 2.89076}
hl_b {6.25: 2.89045, 12.5: 2.8903, 17.7: 2.89038, 25.0: 2.8904, 35.4: 2.89048, 50.0: 2.89063, 100.0: 2.89104, 200.0: 2.89131}
k_b {10.0: 2.89063, 17.3: 2.8903, 30.0: 2.89038, 54.8: 2.89065, 100.0: 2.89103, 300.0: 2.89178}
1등 vs model_11 이득 ×1000: 2.702 (95% 1.392 ~ 4.131)


#### 선택: 2019 1등 그대로 (반감 42.1, k_hand 19.9, 모든 카운트 δ 반감 12.5 · k_b 17.3, 잔차 기준 반감 없음 = 누적)
- 2019 24칸 log loss (실제 구종): model_11 2.89276 -> 2.89006 (−0.0027, 경기 단위 95% 이득 +1.4 ~ +4.1 ×1000)
- 첫 실행은 세밀한 격자 뒤 범위 끝 확인이 빠져 있었음 (δ 반감 12.5·잔차 기준 반감 1000이 끝) -> 넓혀 다시: δ 반감 6.25는 나빠짐 (12.5가 안쪽 1등), 잔차 기준은 1000 · 3000 · 누적이 0.0001 안 (평평)
- δ 반감 12.5구 (다른 카운트 공 기준, 선발 한 경기 안팎) -> 다른 카운트 위치 성향은 아주 최근 것만 의미 있음
- k_hand 17.3 -> 19.9, 위치 반감 42.1 그대로 / k_type 1000 고정 (평평)

In [16]:
LOC_PICK = {k: (None if pd.isna(best[k]) else float(best[k])) for k in list(GRID_AC) + ["ref_hl"]}  # 2019 1등 (잔차 기준 반감기까지)
LOC_HL, K_HAND = LOC_PICK["half_life"], LOC_PICK["k_hand"]
LOC_AC = (LOC_PICK["ref_hl"], LOC_PICK["hl_b"], LOC_PICK["k_b"])
print(f"위치: 반감 {LOC_HL}, k_type {K_TYPE}, k_hand {K_HAND}, 시즌 초 배수 {LOC_SEASON_F}, 모든 카운트 (잔차 기준 반감, δ 반감, k_b) = {LOC_AC}")

위치: 반감 42.1, k_type 1000.0, k_hand 19.9, 시즌 초 배수 0.707, 모든 카운트 (잔차 기준 반감, δ 반감, k_b) = (None, 12.5, 17.3)


### 3-2. 진단: 어디서 좋아졌나 (model_11 vs model_12, 실제 구종 기준) + 구종별 위치 차이 기대 vs 실제

- 이득 = model_11 − model_12 공별 24칸 log loss ×1000 (양수 = 좋아짐), 2019 판단 / 2020 확인
- 그룹: 투수×구종×손 전날까지 초구 수 (초구 기록이 적은 구종에서 다른 카운트를 빌리는 효과가 커야 함), 다른 카운트 기록 수, 구종, 그 시즌 몇 번째 등판
- 4-23 "구종을 몰라서 잃는 몫": 모델이 기대한 몫(구종과 칸의 상호정보) > 실제로 잃은 몫 -> 모든 카운트로 드문 구종 분포가 안정되면 차이가 줄어야 함

In [17]:
app2 = m.drop_duplicates(["pitcher_code", "gameID"])[["pitcher_code", "gameID", "game_date", "GYEAR"]].sort_values(["game_date", "gameID"])
app2["app_no"] = app2.groupby(["pitcher_code", "GYEAR"]).cumcount() + 1  # 그 시즌 몇 번째 등판 (4-23과 같음)
_, NH2 = fc.lookup(fc.prior_table(fp, K2, ["x"]), fp, K2, ["x"])  # 투수×구종×손 전날까지 초구 수 (가중 없음)
LOC15 = dict(half_life=42.1, k_type=1000.0, k_hand=17.3, ref=LOC_REF, ref_hl=LOC_REF_HL, season_f=0.707, ac=None)  # model_11 위치
LOC16 = dict(half_life=LOC_HL, k_type=K_TYPE, k_hand=K_HAND, ref=LOC_REF, ref_hl=LOC_REF_HL, season_f=LOC_SEASON_F, ac=LOC_AC)
rows, lost = [], {}
for yr, q, msk in [(fc.VAL_YEAR, q_val, val), (fc.TEST_YEAR, q_test, test)]:
    i, yz = np.arange(len(q)), q["zone"].map(fc.ZONE_IDX).to_numpy()
    nll_ = lambda P: -np.log(np.clip(P[i, yz], 1e-12, None))
    Pt = P_TYPE_ALL[msk]  # 구종 확률 (model_09)
    for nm, L in [("model_11", LOC15), (MODEL, LOC16)]:  # 구종을 몰라서 잃는 몫
        PZall = np.stack([loc_given_type(q, yr, t, **L)[0] for t in CLASSES], 1)  # 구종 9개 각각 가정 (n, 9, 24)
        Pmix = np.einsum("nt,ntz->nz", Pt, PZall)
        ll_t = nll_(PZall[i, q["pitch_type"].map(fc.TYPE_IDX).to_numpy()])
        kl = (Pt * (PZall * (np.log(np.clip(PZall, 1e-12, None)) - np.log(np.clip(Pmix, 1e-12, None))[:, None, :])).sum(2)).sum(1)
        lost[(yr, nm)] = {"실제 구종 ll": ll_t.mean(), "실전 ll": nll_(Pmix).mean(), "실제로 잃은 몫": (nll_(Pmix) - ll_t).mean(), "모델이 기대한 몫": kl.mean()}
        if nm == MODEL:
            ll16 = ll_t
    ll15 = fc.align(m11, fp, yr)["ll_zone_true"].to_numpy()
    _, Nac = ac_sn(q, LOC_AC[0], LOC_AC[1], 1.0)  # 다른 카운트 기록량 (반감 hl_b 무게 합, 시즌 초 배수 없이)
    d = pd.DataFrame({"이득": (ll15 - ll16) * 1000,
                      "초구 기록 수": pd.cut(NH2[fp["GYEAR"].eq(yr).to_numpy(), 0], [0, 1, 31, 101, 301, 1e9], right=False, labels=["0", "1–30", "31–100", "101–300", "301+"]),
                      "다른 카운트 기록 (반감 무게)": pd.cut(Nac[:, 0], [0, 1, 10, 30, 1e9], right=False, labels=["0", "1–10", "10–30", "30+"]),
                      "구종": q["pitch_type"].to_numpy(),
                      "등판": pd.cut(q[["pitcher_code", "gameID"]].merge(app2, how="left")["app_no"].to_numpy(), [1, 2, 4, 11, 999], right=False, labels=["1", "2–3", "4–10", "11+"])})
    for var in ["초구 기록 수", "다른 카운트 기록 (반감 무게)", "구종", "등판"]:
        for v, g in d.groupby(var, observed=True):
            rows.append({"연도": yr, "변수": var, "값": str(v), "공": len(g), "이득 ×1000": g["이득"].mean()})
    rows.append({"연도": yr, "변수": "전체", "값": "", "공": len(d), "이득 ×1000": d["이득"].mean()})
display(pd.DataFrame(rows).set_index(["변수", "값", "연도"]).unstack("연도").round(2))
display(pd.DataFrame(lost).T.round(4))

공          이득 ×1000        
연도                            2019     2020     2019    2020
변수                값                                         
구종                CHUP      1654.0   1689.0     5.64    6.66
                  CURV      1190.0    951.0     2.19    0.70
                  CUTT        57.0    189.0    23.86    4.31
                  FAST      6627.0   6047.0     1.79    1.08
                  FORK       720.0    532.0     1.43    6.95
                  KNUC         NaN     17.0      NaN  334.35
                  SLID      2458.0   2434.0     1.21    2.93
                  TWOS       752.0    756.0     9.60    1.55
다른 카운트 기록 (반감 무게) 0           61.0     29.0    -0.24   -0.00
                  10–30    13155.0  12529.0     3.06    2.57
                  1–10       242.0     57.0   -15.94   82.92
등판                1          419.0    455.0     2.43    7.77
                  11+       8741.0   8106.0     2.95    2.89
                  2–3        925.0    934.0     2.56    1.97
                  4–10      3373.0   3120.0     2.14    2.61
전체                         13458.0  12615.0     2.70    2.93
초구 기록 수           0           89.0     34.0    -8.71   -2.42
                  101–300   5672.0   5826.0     3.79    1.50
                  1–30      1620.0    729.0     6.31   16.93
                  301+      2461.0   3671.0    -0.90    1.45
                  31–100    3616.0   2355.0     2.11    4.52

실제 구종 ll   실전 ll  실제로 잃은 몫  모델이 기대한 몫
2019 model_11    2.8928  2.9491    0.0563     0.0599
     model_12    2.8901  2.9480    0.0579     0.0622
2020 model_11    2.8963  2.9453    0.0490     0.0598
     model_12    2.8934  2.9448    0.0514     0.0634

#### 🔍 3-2 결과: 초구 기록 적은 구종에서 크게 좋아짐, 구종별 위치 차이 기대 > 실제는 그대로
- 전체 이득 ×1000 (실제 구종): 2019 +2.70, 2020 +2.93
- 초구 기록 수 (투수×구종×손): 1–30구 +6.3 / +16.9, 31–100구 +2.1 / +4.5, 101–300구 +3.8 / +1.5, 301구 이상 −0.9 / +1.5 -> 초구 기록이 적을수록 다른 카운트를 빌리는 효과가 큼 (예상대로)
  - 기록 0구는 −8.7 / −2.4 (공 89 · 34개)
- 구종: 체인지업 +5.6 / +6.7, 투심 +9.6 / +1.6, 직구 +1.8 / +1.1 -> 모든 구종에서 +
- 등판 순번: 고르게 + (첫 등판 +2.4 / +7.8)
- 구종을 몰라서 잃는 몫: 실제 0.056 -> 0.058 / 0.049 -> 0.051, 기대 0.060 -> 0.062 / 0.060 -> 0.063 -> 차이(기대 > 실제)는 그대로
  - 구종별 위치 분포가 더 달라짐 -> 실전 개선(2019 −0.0011)이 실제 구종 기준(−0.0027)의 약 40% (model_01 3-2와 같은 구조)
  - 4-23의 원인 추정(드문 구종 분포의 표본 잡음)은 이 방법으로는 확인 안 됨

## 4. 실전 예측 (예측 구종으로 위치 이어서)

- model_11와 같음, 위치만 3-1에서 고른 값 (모든 카운트 포함)

In [18]:
def predict(q, year, Pt):  # 초구 q의 공별 예측·점수
    n, i = len(q), np.arange(len(q))
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()  # 실제 구종·칸 번호
    nll = lambda p: -np.log(np.clip(p, 1e-12, None))  # 공별 log loss (prob_metrics와 같은 방식으로 자름)

    # ① 구종 확률 (n, 9): 2장 결과를 받음

    # ② 위치: 구종 9개 각각 가정 -> 24칸 확률, 실제 위치의 로그 밀도, 예측 중심 (좌우, 높이)
    PZ, LD, MX, MZ = [], [], [], []
    for t in CLASSES:
        P, g = loc_given_type(q, year, t, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL, LOC_SEASON_F, LOC_AC)
        PZ.append(P); LD.append(fc.log_dens(g, q)); MX.append(g[0]); MZ.append(g[1])
    PZ, LD, MX, MZ = np.stack(PZ, 1), np.stack(LD, 1), np.stack(MX, 1), np.stack(MZ, 1)  # (n, 구종, ...)
    joint = Pt[:, :, None] * PZ  # 구종 확률 × 그 구종일 때 칸 확률 = 구종×칸 결합 확률 (n, 9, 24)
    P_zone = joint.sum(1)  # 실전 24칸 확률 = 구종으로 합침
    # 좌표: 구종 확률로 섞은 정규분포에서 실제 위치의 로그 밀도
    ld_mix = np.log(np.einsum("nt,nt->n", Pt, np.exp(LD - LD.max(1, keepdims=True)))) + LD.max(1)  #  공마다 최댓값 빼기 -> exp -> 섞기 -> log (계산 안전)
    dist_true = np.hypot(q["x"] - MX[i, yt], q["CROSS_PLATE_Z_CM"] - MZ[i, yt])  # 실제 구종 중심과의 거리 cm
    dist_mix = np.hypot(q["x"] - (Pt * MX).sum(1), q["CROSS_PLATE_Z_CM"] - (Pt * MZ).sum(1))  # 구종 확률로 가중한 중심과의 거리 cm

    t_hat = np.array(CLASSES)[Pt.argmax(1)]  # 1등 구종 이름

    # 공별 결과 표 (필수: 키, 구종 확률 9개, 실전 24칸 확률 / 나머지는 선택 열)
    return pd.concat([
        q[["gameID", "pit_no", "pitcher_code", "GYEAR", "pitch_type", "zone"]].reset_index(drop=True),
        pd.DataFrame(Pt, columns=fc.P_TYPE),  # p_type_<구종>
        pd.DataFrame(P_zone, columns=fc.P_ZONE),  # p_zone_<칸>
        pd.DataFrame({"t_hat": t_hat}),  # 1등 구종 이름
        pd.DataFrame({
            "ll_type": nll(Pt[i, yt]),  # 구종 log loss
            "ll_zone": nll(P_zone[i, yz]),  # 24칸 log loss (실전)
            "ll_zone_true": nll(PZ[i, yt, yz]),  # 24칸 log loss (실제 구종)
            "ll_joint": nll(joint[i, yt, yz]),  # 구종×칸 log loss
            "nll_coord": -ld_mix,  # 좌표 NLL (실전)
            "nll_coord_true": -LD[i, yt],  # 좌표 NLL (실제 구종)
            "dist": np.asarray(dist_mix),  # 좌표 거리 cm (실전)
            "dist_true": np.asarray(dist_true),  # 좌표 거리 cm (실제 구종)
        })], axis=1)

In [19]:
# 2019 검증 + 2020 평가 초구 전부 예측
pp = pd.concat([predict(q_val, fc.VAL_YEAR, P_TYPE_ALL[val]), predict(q_test, fc.TEST_YEAR, P_TYPE_ALL[test])], ignore_index=True)
print(pp.shape)
# 확인: 위치와 무관한 열(구종)은 model_11와 같아야 함
chk = pp.merge(m11, on=fc.KEYS, suffixes=("", "_11"))
same = ["ll_type"] + fc.P_TYPE
bad = [c for c in same if not np.allclose(chk[c], chk[c + "_11"], equal_nan=True, atol=1e-6)]
assert len(chk) == len(pp) and not bad, f"model_11와 다른 열: {bad}"
print("구종 = model_11 (재현 확인), 위치만 바뀜")

(26073, 48)
구종 = model_11 (재현 확인), 위치만 바뀜


## 5. 평가

- 채점은 `fc.evaluate`, model_11와 나란히 / 경기 단위 부트스트랩 (model_12 − model_11, 2020)
- 2020은 확인용 -> 2020을 보고 파라미터 다시 고르지 않음

In [20]:
def eval_years(pp_):  # 2019·2020 주요 지표 표
    return pd.DataFrame({"검증 2019": fc.evaluate(pp_, fp, fc.VAL_YEAR), "평가 2020": fc.evaluate(pp_, fp, fc.TEST_YEAR)})

summary = eval_years(pp)
m11_sum = eval_years(m11)
display(pd.concat({MODEL: summary, "model_11": m11_sum, "차이 (model_11 대비)": summary - m11_sum}, axis=1).round(4))

model_12          model_11          차이 (model_11 대비)        
                     검증 2019  평가 2020  검증 2019  평가 2020          검증 2019 평가 2020
구종 logloss            1.0613   1.0561   1.0613   1.0561           0.0000  0.0000
구종 top1               0.5385   0.5332   0.5385   0.5332           0.0000  0.0000
구종 top2               0.8177   0.8218   0.8177   0.8218           0.0000  0.0000
구종 top3               0.9374   0.9412   0.9374   0.9412           0.0000  0.0000
구종 macro F1           0.3226   0.3545   0.3226   0.3545           0.0000  0.0000
24칸 logloss (실전)      2.9480   2.9448   2.9491   2.9453          -0.0011 -0.0005
24칸 top1              0.1314   0.1321   0.1303   0.1331           0.0012 -0.0010
24칸 top3              0.2893   0.2917   0.2903   0.2920          -0.0010 -0.0003
H/S/C logloss         1.0751   1.0765   1.0747   1.0765           0.0004 -0.0000
H/S/C 정확도             0.3942   0.3910   0.3953   0.3898          -0.0011  0.0013
24칸 logloss (실제 구종)   2.8901   2.8934   2.8928   2.8963          -0.0027 -0.0029
구종×칸 logloss          3.9513   3.9489   3.9541   3.9518          -0.0027 -0.0029
좌표 평균거리 cm           32.6480  31.8177  32.6727  31.8253          -0.0247 -0.0076
좌표 NLL                9.3152   9.2657   9.3168   9.2661          -0.0016 -0.0004

In [21]:
display(fc.bootstrap_diff(fc.pitch_scores(m11, fp), fc.pitch_scores(pp, fp)).round(4))  # 음수 = 이 모델이 좋음 (b − a)

,차이 (b−a),95% 하한,95% 상한,판정
ll_type,0.0000,0.0000,0.0000,차이 불확실
ll_zone,-0.0005,-0.0014,0.0003,차이 불확실
ll_joint,-0.0029,-0.0047,-0.0015,좋아짐
nll_coord,-0.0004,-0.0015,0.0006,차이 불확실
dist,-0.0076,-0.0234,0.0081,차이 불확실


### 🥇1등🥇
#### 2020 24칸 log loss 실제 구종 2.8963 -> 2.8934 (−0.0029), 실전 2.9453 -> 2.9448, 구종×칸 좋아짐, 나빠진 점수 없음 -> 채택
- 구종은 model_11와 같음 (재현 확인) -> 실전 변화 = 순수한 위치 효과
- 2019 실제 구종 −0.0027 (95% 이득 +1.4 ~ +4.1 ×1000) -> 2020 −0.0029로 그대로 이어짐 (시즌 초 배수 −0.0005의 약 5배)
- 부트스트랩 (2020, model_12 − model_11): 구종×칸 −0.0029 (95% −0.0047 ~ −0.0015) 좋아짐, 24칸 −0.0005 · 좌표 NLL −0.0004 · 거리 −0.008cm 차이 불확실
- 실전 개선이 실제 구종 기준의 약 40% (구종별 위치 분포가 더 달라져 구종을 틀리면 잃는 몫도 커짐, 4-2)
- H/S/C 정확도 2019 −0.0011 / 2020 +0.0013 (거의 그대로)
- **채택 -> 다음 비교 출발점** (사용자 결정 2026-11-10)

## 6. 저장

- `results/model_12/` (채택하면 `compare.ipynb` `ADOPTED`에 추가)

In [22]:
fc.save_results(MODEL, pp, {"summary": summary})

저장: results\model_12 ['per_pitch.parquet', 'summary.csv']
